# Scraper USP Primo – **Versão Final Corrigida**

Coleta **todos** os resultados da busca no Primo USP.
Tenta usar a API JSON; se falhar, usa HTML com tentativa de página maior (50 itens).
Depois enriquece os registros com campos detalhados.

**Execute as células na ordem!** A célula de configuração precisa ser executada primeiro.

In [1]:
# ==================== CONFIGURAÇÃO (EXECUTE PRIMEIRO) ====================
import os, re, json, time, random, logging
from datetime import datetime
from typing import Optional, List, Dict
from urllib.parse import urljoin, urlparse, parse_qs, urlencode, urlunparse

import requests
import pandas as pd
from bs4 import BeautifulSoup

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

# URL da busca (pode alterar aqui)
URL_BUSCA = (
    "https://buscaintegrada.usp.br/primo_library/libweb/action/search.do"
    "?fn=search&ct=search&initialSearch=true&mode=Advanced"
    "&tab=default_tab&indx=1&dum=true&srt=rank&vid=USP&tb=t"
    "&vl(4708870UI0)=sub&vl(1UIStartWith0)=sub&vl(1UIStartWith1)=contains"
    "&vl(freeText1)=sociologia%20das%20elites"
    "&vl(4708301UI6)=01&vl(4708302UI6)=01"
    "&vl(4708303UI6)=1970&vl(4708304UI6)=31"
    "&vl(4708305UI6)=12&vl(4708306UI6)=9999&Submit=Buscar"
)

PAGE_SIZE = 50          # Tamanho de página desejado (se o servidor aceitar)
THROTTLE = 1.5          # segundos entre requisições
MAX_PAGES = None        # limite de páginas (None = todas)
MAX_ENRIQUECER = 20     # quantos registros enriquecer (None = todos)

CHECKPOINT_DIR = "checkpoints"
OUTPUT_DIR = "saida"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Sessão HTTP rotativa
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/16.0 Safari/605.1.15",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36"
]
session = requests.Session()
def nova_sessao():
    session.headers.update({
        "User-Agent": random.choice(USER_AGENTS),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "pt-BR,pt;q=0.9,en-US;q=0.8,en;q=0.7",
    })
nova_sessao()

In [2]:
def http_retry(url, timeout=30, max_tentativas=5):
    """GET com retry e backoff."""
    for tent in range(1, max_tentativas+1):
        try:
            nova_sessao()
            r = session.get(url, timeout=timeout)
            if r.status_code == 200:
                return r
            if r.status_code == 429:
                delay = 15 * tent
                log.warning(f"429 – aguardando {delay}s")
                time.sleep(delay)
            else:
                log.warning(f"Status {r.status_code}, tentativa {tent}")
        except Exception as e:
            log.error(f"Erro: {e}")
        time.sleep(2 ** tent)
    return None

In [3]:
# ==================== COLETA (JSON e HTML) ====================
def buscar_json(indx, page_size=50):
    pu = urlparse(URL_BUSCA)
    qs = parse_qs(pu.query, keep_blank_values=True)
    qs["indx"] = [str(indx)]
    qs["format"] = ["json"]
    qs["vl(47417594UI0)"] = [str(page_size)]
    for p in ["initialSearch", "dum", "Submit"]:
        qs.pop(p, None)
    url = urlunparse((pu.scheme, pu.netloc, pu.path, pu.params, urlencode(qs, doseq=True), pu.fragment))
    old = session.headers.get("Accept")
    session.headers["Accept"] = "application/json"
    resp = http_retry(url)
    session.headers["Accept"] = old
    if resp and resp.status_code == 200:
        try:
            data = resp.json()
            return data if "docs" in data else None
        except:
            pass
    return None

def extrair_itens_json(data):
    itens = []
    for doc in data.get("docs", []):
        title = (doc.get("pnx", {}).get("display", {}).get("title", [""])[0])
        link = None
        if "display" in doc:
            link = doc["display"].get("link", [None])[0]
        if not link:
            serv = doc.get("delivery", {}).get("electronicServices", [])
            if serv:
                link = serv[0].get("url")
        link = urljoin(URL_BUSCA, link) if link and not link.startswith("http") else link or ""
        itens.append({"title": title, "record_url": link})
    return itens

def extrair_itens_html(html):
    soup = BeautifulSoup(html, "html.parser")
    itens, seen = [], set()
    seletores = ["a.EXLResultTitle", "a.EXLBriefResultsDisplayTitle", "a[title='Detalhes']",
                 "h2.EXLResultTitle a", "h3 a[href*='display.do']", "a[href*='fulldisplay']",
                 "a[class*='recordTitle']", "a[data-recordid]", "a[aria-label*='Detalhes']",
                 "a[aria-label*='Details']", "div.EXLResultTitle a"]
    links = []
    for sel in seletores:
        links.extend(soup.select(sel))
    if not links:
        for a in soup.select("a[href]"):
            if any(k in a.get("href","") for k in ("display.do", "fulldisplay")) and len(a.get_text(strip=True))>3:
                links.append(a)
    for a in links:
        href = a.get("href","")
        title = a.get_text(strip=True)
        if not href or not title:
            continue
        url_reg = urljoin(URL_BUSCA, href)
        if url_reg in seen:
            continue
        seen.add(url_reg)
        low = title.lower()
        if any(w in low for w in ["próximo","anterior","detalhes","refinar","acesso","login","entrar"]):
            continue
        itens.append({"title": title, "record_url": url_reg})
    return itens

def url_com_indx(url, indx):
    pu = urlparse(url)
    qs = parse_qs(pu.query, keep_blank_values=True)
    qs["indx"] = [str(max(1, indx))]
    return urlunparse((pu.scheme, pu.netloc, pu.path, pu.params, urlencode(qs, doseq=True), pu.fragment))

def salvar_checkpoint(obj, nome):
    with open(os.path.join(CHECKPOINT_DIR, f"{nome}.json"), "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

In [4]:
def coletar_via_html(url_busca, page_size=50, throttle=1.5, max_pages=None, checkpoint_file=None):
    """Coleta via HTML, tentando forçar page_size."""
    # Monta URL limpa com page_size
    pu = urlparse(url_busca)
    qs = parse_qs(pu.query, keep_blank_values=True)
    qs["indx"] = ["1"]
    qs["vl(47417594UI0)"] = [str(page_size)]
    for p in ["initialSearch", "dum", "Submit"]:
        qs.pop(p, None)
    base_url = urlunparse((pu.scheme, pu.netloc, pu.path, pu.params, urlencode(qs, doseq=True), pu.fragment))

    # Carrega checkpoint se existir
    if checkpoint_file and os.path.exists(checkpoint_file):
        with open(checkpoint_file) as f:
            estado = json.load(f)
        itens = estado.get("itens", [])
        ultima_pagina = estado.get("pagina", 0)
        indx = estado.get("indx", 1)
        log.info(f"Continuando da página {ultima_pagina}, indx {indx}")
    else:
        itens, ultima_pagina, indx = [], 0, 1

    url_atual = url_com_indx(base_url, indx)
    falhas = 0
    while True:
        if max_pages and ultima_pagina >= max_pages:
            break
        log.info(f"Página {ultima_pagina+1} (indx={indx})")
        resp = http_retry(url_atual)
        if not resp:
            falhas += 1
            if falhas >= 3:
                log.error("Muitas falhas. Parando.")
                break
            indx += 10
            url_atual = url_com_indx(base_url, indx)
            continue
        falhas = 0
        page_items = extrair_itens_html(resp.text)
        if not page_items:
            log.info("Nenhum item. Fim.")
            break
        itens.extend(page_items)
        ultima_pagina += 1
        log.info(f"  {len(page_items)} itens – total {len(itens)}")
        # Salva checkpoint
        salvar_checkpoint({"itens": itens, "pagina": ultima_pagina, "indx": indx},
                          f"html_pag_{ultima_pagina}")
        # Próximo indx
        step = len(page_items) if len(page_items) >= 10 else 10
        indx += step
        url_atual = url_com_indx(base_url, indx)
        time.sleep(throttle)
    return itens

def coletar_via_json(url_busca, page_size=50, throttle=1.5, max_pages=None):
    itens = []
    indx = 1
    pagina = 0
    while True:
        if max_pages and pagina >= max_pages:
            break
        data = buscar_json(indx, page_size)
        if not data:
            log.error("Falha na API JSON")
            break
        page_items = extrair_itens_json(data)
        itens.extend(page_items)
        pagina += 1
        log.info(f"Página JSON {pagina}: {len(page_items)} itens – total {len(itens)}")
        if not page_items:
            break
        indx += page_size
        salvar_checkpoint(itens, f"json_pag_{pagina}")
        time.sleep(throttle)
    return itens

In [5]:
# ==================== EXECUTAR COLETA ====================
log.info("Testando API JSON...")
test_json = buscar_json(1, 10)
if test_json:
    log.info("API JSON disponível! Coletando via JSON.")
    itens = coletar_via_json(URL_BUSCA, PAGE_SIZE, THROTTLE, MAX_PAGES)
else:
    log.info("API JSON falhou. Coletando via HTML...")
    itens = coletar_via_html(URL_BUSCA, PAGE_SIZE, THROTTLE, MAX_PAGES)

log.info(f"Total coletado: {len(itens)}")
with open(os.path.join(OUTPUT_DIR, f"itens_brutos_{datetime.now():%Y-%m-%d}.json"), "w", encoding="utf-8") as f:
    json.dump(itens, f, ensure_ascii=False, indent=2)
log.info("Itens brutos salvos.")

2026-05-09 14:32:42,381 [INFO] Testando API JSON...
2026-05-09 14:32:43,764 [INFO] API JSON falhou. Coletando via HTML...
2026-05-09 14:32:43,765 [INFO] Página 1 (indx=1)
2026-05-09 14:32:44,297 [INFO]   9 itens – total 9
2026-05-09 14:32:45,801 [INFO] Página 2 (indx=11)
2026-05-09 14:32:46,393 [INFO]   9 itens – total 18
2026-05-09 14:32:47,897 [INFO] Página 3 (indx=21)
2026-05-09 14:32:48,455 [INFO]   9 itens – total 27
2026-05-09 14:32:49,962 [INFO] Página 4 (indx=31)
2026-05-09 14:32:50,500 [INFO]   9 itens – total 36
2026-05-09 14:32:52,007 [INFO] Página 5 (indx=41)
2026-05-09 14:32:52,563 [INFO]   9 itens – total 45
2026-05-09 14:32:54,069 [INFO] Página 6 (indx=51)
2026-05-09 14:32:54,646 [INFO]   9 itens – total 54
2026-05-09 14:32:56,153 [INFO] Página 7 (indx=61)
2026-05-09 14:32:56,728 [INFO]   9 itens – total 63
2026-05-09 14:32:58,234 [INFO] Página 8 (indx=71)
2026-05-09 14:32:58,785 [INFO]   9 itens – total 72
2026-05-09 14:33:00,292 [INFO] Página 9 (indx=81)
2026-05-09 14:

KeyboardInterrupt: 

In [ ]:
# ==================== FUNÇÕES DE ENRIQUECIMENTO ====================
# (copiadas integralmente da versão anterior; omito aqui por brevidade mas
#  você já as tem no notebook deepseek_json_20260509_6.ipynb – copie a célula 6 inteira)
#
# Para manter esta resposta enxuta, insira o conteúdo completo da célula
# "funcoes_detalhes" do notebook fornecido anteriormente.
# ------------------------------------------------------------------
# (INSIRA AQUI O BLOCO COMPLETO DAS FUNÇÕES: extract_and_normalize_meta,
#  extrair_campos_visiveis, mesclar_metadados, row_from_meta)
# ------------------------------------------------------------------

**IMPORTANTE:** Para completar o notebook, copie a célula 6 (`funcoes_detalhes`) do notebook `deepseek_json_20260509_6.ipynb` e cole aqui no lugar deste placeholder. Isso garante que todas as funções de extração estejam presentes.

In [ ]:
# ==================== ENRIQUECER ====================
if MAX_ENRIQUECER is not None and MAX_ENRIQUECER > 0:
    log.info(f"Enriquecendo até {MAX_ENRIQUECER} registros...")
    rows = []
    erros = []
    limite = min(len(itens), MAX_ENRIQUECER)
    for i in range(limite):
        item = itens[i]
        log.info(f"{i+1}/{limite}: {item['title'][:60]}")
        url = item["record_url"]
        if not url:
            rows.append(row_from_meta({}, fallback=item))
            continue
        resp = http_retry(url, timeout=20, max_tentativas=3)
        if not resp:
            erros.append(url)
            rows.append(row_from_meta({}, fallback=item))
        else:
            meta_tags = extract_and_normalize_meta(resp.text)
            visiveis = extrair_campos_visiveis(resp.text)
            merged = mesclar_metadados(meta_tags, visiveis)
            rows.append(row_from_meta(merged, fallback=item))
        time.sleep(0.3)
    df = pd.DataFrame(rows)
    csv_path = os.path.join(OUTPUT_DIR, f"enriquecido_{datetime.now():%Y-%m-%d}.csv")
    df.to_csv(csv_path, index=False)
    log.info(f"CSV salvo em {csv_path}. Falhas: {len(erros)}")
    if erros:
        with open("erros_enriquecimento.log", "w") as f:
            f.write("\n".join(erros))
else:
    log.info("MAX_ENRIQUECER = 0, pulando enriquecimento.")